In [1]:
import sys
# 프로젝트 루트 디렉토리를 sys.path에 추가
root_path = r"D:\finalProjectB\back\apcms_ai"
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

In [2]:
#DB 조회 쿼리로 추출한 CSV파일 확인

In [3]:
from pathlib import Path
import pandas as pd
import os

print(os.getcwd())
#D:\finalProjectB\back\apcms_ai\ai_recmn\testPerModule

CSV = Path("../data/parking_history.csv")
# T or F
assert CSV.exists(), f"SQL to {CSV} 실패한 듯?"

raw = pd.read_csv(CSV)
print("전체 건수:", len(raw))
# group by(OFFICE_ID, TARGET_ZONE) + Null 결측치 처리 결과
print(raw.groupby(["OFFICE_ID", "TARGET_ZONE"]).size().unstack(fill_value=0))
raw.head(10)

D:\finalProjectB\back\apcms_ai\ai_recmn\testPerModule
전체 건수: 450
TARGET_ZONE    A    B    C
OFFICE_ID                 
1            105   30   15
2             15  105   30
3             30   15  105


,USAGE_ID,OFFICE_ID,USAGE_DATE,TARGET_ZONE
0,1,1,2026-05-04 09:15:00,A
1,151,2,2026-05-04 09:15:00,B
2,301,3,2026-05-04 09:15:00,C
3,2,1,2026-05-05 10:30:00,A
4,152,2,2026-05-05 10:30:00,B
5,302,3,2026-05-05 10:30:00,C
6,3,1,2026-05-06 08:45:00,A
7,153,2,2026-05-06 08:45:00,B
8,303,3,2026-05-06 08:45:00,C
9,4,1,2026-05-07 09:00:00,A


In [4]:
# 과거 이용이력을 기반으로 학습에 필요한 특성 추출

In [6]:
# 각 행의 정답을 알려주지 않고 해당 시점 이전의 기록만으로 구역별 이용 비율과 최근 10회 빈도 생성. 월=0, 일=6
from ai_recmn.training.features import FEATURES, prepare_training_data
print(FEATURES)
df = prepare_training_data(raw) # 내부에서 컬럼명 소문자로 변환
df[["office_id", "usage_date", "target_zone"] + FEATURES].head(12)

['day_of_week', 'entry_hour', 'is_weekend', 'pref_score_A', 'pref_score_B', 'pref_score_C', 'recent_A_count', 'recent_B_count', 'recent_C_count']


,office_id,usage_date,target_zone,day_of_week,entry_hour,is_weekend,pref_score_A,pref_score_B,pref_score_C,recent_A_count,recent_B_count,recent_C_count
0,1,2026-05-04 09:15:00,A,0,9,0,0.333333,0.333333,0.333333,0,0,0
1,1,2026-05-05 10:30:00,A,1,10,0,1.000000,0.000000,0.000000,1,0,0
2,1,2026-05-06 08:45:00,A,2,8,0,1.000000,0.000000,0.000000,2,0,0
3,1,2026-05-07 09:00:00,A,3,9,0,1.000000,0.000000,0.000000,3,0,0
4,1,2026-05-08 10:15:00,A,4,10,0,1.000000,0.000000,0.000000,4,0,0
5,1,2026-05-09 08:30:00,A,5,8,1,1.000000,0.000000,0.000000,5,0,0
6,1,2026-05-10 09:45:00,A,6,9,1,1.000000,0.000000,0.000000,6,0,0
7,1,2026-05-11 20:00:00,B,0,20,0,1.000000,0.000000,0.000000,7,0,0
8,1,2026-05-12 18:15:00,B,1,18,0,0.875000,0.125000,0.000000,7,1,0
9,1,2026-05-13 13:30:00,C,2,13,0,0.777778,0.222222,0.000000,7,2,0


In [ ]:
# 학습 : 평가 데이터 분리 (시간순 8 : 2)

In [7]:
# 미리 데이터를 섞어 학습하지 않도록 과거 날짜 80%에서 학습하고 이후 20%에서 평가

# 주차이력의 날짜에서 시간 정보를 제거하고, 중복 없는 날짜를 오름차순으로 정렬
days = sorted(df["usage_date"].dt.normalize().unique())

# 전체 날짜의 80% 지점을 학습/평가 데이터 분리 기준일로 설정 (8:2 비율)
# 행 개수가 아닌 고유 날짜 개수의 80%를 기준으로 분리
# train_test_split()과 달리 날짜순으로 분리한 이유는 과거 주차이력으로 학습하고 미래 주차이력을 예측하는 실제 예약 시스템 환경을 재현하기 위함
boundary = days[int(len(days) * 0.8)]

# 기준일 이전의 주차이력을 학습 데이터(Train)로 분리
train_df = df[df["usage_date"] < boundary]

# 기준일 이후(기준일 포함)의 주차이력을 평가 데이터(Test)로 분리
test_df = df[df["usage_date"] >= boundary]

# 학습 및 평가 데이터 건수와 평가 시작일 출력
print("학습:", len(train_df), "평가:", len(test_df), "평가 시작일:", boundary)

학습: 360 평가: 90 평가 시작일: 2026-09-01 00:00:00


In [ ]:
# RandomForest 학습 및 기준선 비교

In [8]:
from ai_recmn.training.train import train
metrics = train(CSV)  # models/parking_rf.joblib 저장
print("Random Forest:", metrics["random_forest_accuracy"])
print("최빈 구역 기준선:", metrics["most_used_zone_accuracy"])

{
  "train_rows": 360,
  "test_rows": 90,
  "test_from": "2026-09-01",
  "random_forest_accuracy": 0.9333,
  "most_used_zone_accuracy": 0.7,
  "classification_report": {
    "A": {
      "precision": 0.9032258064516129,
      "recall": 0.9333333333333333,
      "f1-score": 0.9180327868852459,
      "support": 30.0
    },
    "B": {
      "precision": 0.9642857142857143,
      "recall": 0.9,
      "f1-score": 0.9310344827586207,
      "support": 30.0
    },
    "C": {
      "precision": 0.9354838709677419,
      "recall": 0.9666666666666667,
      "f1-score": 0.9508196721311475,
      "support": 30.0
    },
    "accuracy": 0.9333333333333333,
    "macro avg": {
      "precision": 0.934331797235023,
      "recall": 0.9333333333333335,
      "f1-score": 0.9332956472583381,
      "support": 90.0
    },
    "weighted avg": {
      "precision": 0.934331797235023,
      "recall": 0.9333333333333333,
      "f1-score": 0.9332956472583379,
      "support": 90.0
    }
  }
}
Random Forest: 0.9333


In [ ]:
# 실제 서비스 예측 모듈 점검

In [9]:
# 모델은 학습 시점에 저장한 파일을 불러오며 서비스 API에서도 해당 함수 호출.
from ai_recmn.service.predictor import recommend
sample_history = [
    {"entry_at": row.usage_date, "zone_code": row.target_zone}
    for row in df[df["office_id"] == 1].itertuples()
]
recommend("2026-10-20T09:00:00", sample_history)

[{'zone_code': 'A', 'score': 0.951097},
 {'zone_code': 'C', 'score': 0.033955},
 {'zone_code': 'B', 'score': 0.014948}]

In [ ]:
# 주의사항: AI와 비교한 최빈 구역 추천은 별도 ML 모델이 아니라 과거 이용 횟수만 활용한 통계 규칙